## LRU Cache for "Activation Checkpointing"

The Challenge: Build a memory-aware cache for model activations.Scenario: During the forward pass, you store activations. If memory usage exceeds a limit, you must "evict" (delete) activations based on an LRU policy.

Requirements: store(layer_id, tensor_data) and retrieve(layer_id).

If retrieve is called on an evicted layer, it must trigger a recompute() function (mocked).The cache must track "memory cost" (e.g., tensor size) rather than just the number of items.Performance Thinking: Discuss $O(1)$ lookup for the cache. Explain why a standard dictionary isn't enough (need a Doubly Linked List + Hash Map) and how you'd handle "pinned" memory that cannot be evicted.

In [ ]:
from collections import OrderedDict

lru_cache = OrderedDict()
MAX_SIZE = 1e7
def store(layer_id, tensor_data):
    if layer_id in lru_cache:
        lru_cache.move_to_end(layer_id)
        lru_cache[layer_id] = tensor_data
        return
    else:
        # Calculate mem required
        mem_required = tensor_data.numel()

        if mem_required > MAX_SIZE:
            raise ValueError("Memory required exceeds max size")
        
        # Evict until we have enough space
        memory_cost = sum([tensor_data.numel() for tensor_data in lru_cache.values()])
        while memory_cost > MAX_SIZE - mem_required:
            evicted_data = lru_cache.popitem(last=False)
            memory_cost -= evicted_data[1].numel()
        
        # store the tensor data
        lru_cache[layer_id] = tensor_data

        # move to end to make it the most recently used
        lru_cache.move_to_end(layer_id) 
        return

def retrieve(layer_id):
    if layer_id not in lru_cache:
        recompute(layer_id)
    else:
        lru_cache.move_to_end(layer_id)
    
    return lru_cache[layer_id]


def recompute(layer_id):
    # Mocked recomputation
    tensor_data = ...
    store(layer_id, tensor_data)
    

